# ZS-BERT Simulation (NAACL-HLT 2021 Reproduction — FewRel $m=10$)

Notebook này thực thi mô phỏng bài báo **"ZS-BERT: Towards Zero-Shot Relation Extraction with Attribute Representation Learning"** (Chih-Yao Chen & Cheng-Te Li, NAACL-HLT 2021) từ repository:
- **GitHub Repository:** `https://github.com/Student-persist-1919/zs-bert-simulation`
- **Cấu hình chuẩn theo bài báo (Section 5.1, trang 6):**
  - Backbone câu: `bert-large-cased` ($d = 1024$)
  - Backbone mô tả quan hệ: `SentenceTransformer('bert-large-nli-mean-tokens')` ($d = 1024$)
  - Số quan hệ Unseen: $m = 10$ (70 seen relations = 49,000 mẫu train; 10 unseen relations = 7,000 mẫu test)
  - Siêu tham số: `batch_size = 4`, `lr = 5e-6` (Adam), `gamma = 7.5`, `alpha = 0.4`, `dist_func = 'inner'`, `epochs = 10`, `seed = 300`
  - Kết quả mục tiêu trong bài báo (Table 2, FewRel $m=10$): **Precision = 56.92%, Recall = 57.59%, F1 = 57.25%**

### 1. Khởi tạo môi trường & Clone Repository từ GitHub
Tự động kiểm tra GPU, cài đặt `sentence-transformers` và clone mã nguồn từ `https://github.com/Student-persist-1919/zs-bert-simulation.git`.

In [ ]:
import os
import shutil
import subprocess
import sys
import torch

os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else os.getcwd()
REPO_URL = "https://github.com/Student-persist-1919/zs-bert-simulation.git"
REPO_DIR = os.path.join(WORK_DIR, "zs-bert-simulation")

print(f"[1/3] Checking PyTorch & GPU environment...", flush=True)
print(f"  Python version : {sys.version.split()[0]}", flush=True)
print(f"  PyTorch version: {torch.__version__}", flush=True)
print(f"  CUDA available : {torch.cuda.is_available()}", flush=True)
if torch.cuda.is_available():
    print(f"  GPU Device     : {torch.cuda.get_device_name(0)}", flush=True)

print("\n[2/3] Installing sentence-transformers...", flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)

print(f"\n[3/3] Cloning {REPO_URL} into {REPO_DIR}...", flush=True)
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.makedirs(os.path.join(REPO_DIR, "data"), exist_ok=True)
print("✅ Environment & repository setup completed!", flush=True)

### 2. Tự động tìm kiếm & Tiền xử lý dữ liệu FewRel
Tìm file `fewrel_dataset.json` trong `/kaggle/input` và chuyển đổi sang `data/fewrel_all.json` (đồng thời chuyển khoảng vị trí `[start, end]` của thực thể sang danh sách chỉ số token chuẩn `list(range(start, end))`).

In [ ]:
import os
import subprocess
import sys

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else os.getcwd()
REPO_DIR = os.path.join(WORK_DIR, "zs-bert-simulation")

def find_dataset_file(filename="fewrel_dataset.json"):
    search_roots = ["/kaggle/input", WORK_DIR, REPO_DIR]
    for root_dir in search_roots:
        if not os.path.exists(root_dir):
            continue
        for dirpath, _, filenames in os.walk(root_dir):
            if filename in filenames:
                return os.path.join(dirpath, filename)
    raise FileNotFoundError(
        f"Không tìm thấy {filename} trong /kaggle/input! Hãy Add Data bộ dữ liệu FewRel vào Kaggle Notebook."
    )

fewrel_input = find_dataset_file("fewrel_dataset.json")
fewrel_output = os.path.join(REPO_DIR, "data", "fewrel_all.json")
preprocess_script = os.path.join(REPO_DIR, "scripts", "preprocess_fewrel.py")

print(f"Found FewRel dataset at: {fewrel_input}", flush=True)
subprocess.run(
    [sys.executable, "-u", preprocess_script, "--input", fewrel_input, "--output", fewrel_output],
    check=True,
)
print("✅ FewRel preprocessing completed!", flush=True)

### 3. Huấn luyện & Đánh giá Zero-Shot trên FewRel ($m=10$)
Chạy `train_fewrel.py` với toàn bộ siêu tham số gốc của bài báo (`--seed 300 --n_unseen 10 --gamma 7.5 --alpha 0.4 --dist_func inner --batch_size 4 --lr 5e-6 --epochs 10`).
- **Streaming Log trực tiếp:** Sử dụng `PYTHONUNBUFFERED=1` và đọc từng dòng qua pipe không đệm (`bufsize=1`), giúp bạn mở tab **Logs** trên Kaggle trong lúc **Save & Commit** để theo dõi trực tiếp từng bước train và kết quả đánh giá sau mỗi epoch.

In [ ]:
import os
import subprocess
import sys

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else os.getcwd()
REPO_DIR = os.path.join(WORK_DIR, "zs-bert-simulation")
MODEL_DIR = os.path.join(REPO_DIR, "model")
LOG_FILE = os.path.join(WORK_DIR, "train_fewrel.log")
SUMMARY_FILE = os.path.join(WORK_DIR, "training_summary_fewrel.json")

cmd = [
    sys.executable,
    "-u",
    "train_fewrel.py",
    "--seed", "300",
    "--n_unseen", "10",
    "--gamma", "7.5",
    "--alpha", "0.4",
    "--dist_func", "inner",
    "--batch_size", "4",
    "--eval_batch_size", "64",
    "--lr", "5e-6",
    "--epochs", "10",
    "--log_every", "500",
    "--fp16",
    "--summary_out", SUMMARY_FILE,
]

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

print(f"🚀 Starting ZS-BERT training (live logs streaming + saving to {LOG_FILE})...\n", flush=True)

process = subprocess.Popen(
    cmd,
    cwd=MODEL_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

with open(LOG_FILE, "w", encoding="utf-8") as log_f:
    for line in process.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        log_f.write(line)
        log_f.flush()

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Training exited with non-zero code: {return_code}")
print("\n✅ Training & Evaluation finished successfully!", flush=True)

### 4. Tổng hợp Kết quả & Đối chiếu với Bài báo NAACL 2021
Đọc file `training_summary_fewrel.json` để hiển thị bảng so sánh chi tiết qua từng epoch và tỷ lệ tái현 (reproduction rate) so với kết quả công bố trong bài báo.

In [ ]:
import json
import os

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else os.getcwd()
SUMMARY_FILE = os.path.join(WORK_DIR, "training_summary_fewrel.json")

if os.path.exists(SUMMARY_FILE):
    with open(SUMMARY_FILE, "r", encoding="utf-8") as f:
        summary = json.load(f)

    print("=" * 75)
    print(f"{'Epoch':^7} | {'Loss':^10} | {'Precision':^11} | {'Recall':^11} | {'F1 Score':^11} | {'% of Paper F1':^14}")
    print("-" * 75)
    for row in summary["history"]:
        print(
            f"{row['epoch']:^7d} | {row['train_loss']:^10.4f} | "
            f"{row['precision']*100:^10.2f}% | {row['recall']*100:^10.2f}% | "
            f"{row['f1']*100:^10.2f}% | {row['paper_f1_similarity_pct']:^13.2f}%"
        )
    print("=" * 75)
    best = summary["best"]
    paper = summary["paper_target"]
    print(
        f"🏆 BEST RESULT (Epoch {best['epoch']}): "
        f"P = {best['precision']*100:.2f}% (Paper: {paper['precision']*100:.2f}%), "
        f"R = {best['recall']*100:.2f}% (Paper: {paper['recall']*100:.2f}%), "
        f"F1 = {best['f1']*100:.2f}% (Paper: {paper['f1']*100:.2f}%)"
    )
    print(f"🎯 Tỷ lệ mô phỏng đạt: {best['paper_f1_similarity_pct']:.2f}% so với bài báo gốc!")
else:
    print("Không tìm thấy file summary.")